# Unknown Edges

This notebook demonstrates the joint optimisation framework for graphs with partially observed edges.

**Key idea:** instead of zero-imputing unknown edges (which biases the embeddings), we jointly optimise the node embeddings *and* the unknown adjacency entries. This is equivalent to choosing the unknown weights that best fit our prior knowledge encoded by the loss function.

In [ ]:
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from hypegrl.embedders.poincare_maps import PoincareMapsEmbedder
from hypegrl.visualization.disk import plot_poincare_graph

## Setup: karate club with unknown edges

In [ ]:
G = nx.karate_club_graph()
all_edges = list(G.edges())

# Treat 10% of edges as unknown
rng = np.random.default_rng(42)
idx = rng.choice(len(all_edges), size=len(all_edges)//10, replace=False)
unknown_edges = [all_edges[i] for i in idx]
print(f'Total edges: {len(all_edges)}, Unknown: {len(unknown_edges)}')

## Biased baseline: zero-imputation

In [ ]:
biased = PoincareMapsEmbedder(d=2, n_steps=300, log_every=0, random_state=0)
biased.fit(G)  # unknown edges get zero weight implicitly

## Joint optimisation

In [ ]:
joint = PoincareMapsEmbedder(
    d=2, n_steps=300, log_every=50, random_state=0,
    regularize_a=0.01,
)
joint.fit(G, unknown_edges=unknown_edges)

## Imputed edge weights

In [ ]:
print('Imputed weights (sigmoid-constrained to (0,1)):')
for (m, n), w in zip(unknown_edges, joint.imputed_weights):
    print(f'  ({m:2d},{n:2d})  estimated={w:.3f}  true=1.0')

## Visualise: biased vs joint

In [ ]:
true_weights = {(min(m,n), max(m,n)): 1.0 for m,n in unknown_edges}

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

plot_poincare_graph(
    G, biased.embeddings(), unknown_edges,
    a_omega_estimated=np.zeros(len(unknown_edges)),
    show_weights=False,
    title='Biased (zero-imputation)',
    ax=axes[0],
)
plot_poincare_graph(
    G, joint.embeddings(), unknown_edges,
    a_omega_estimated=joint.imputed_weights,
    true_weights=true_weights,
    title='Joint optimisation',
    ax=axes[1],
)
fig.savefig('unknown_edges_comparison.png', dpi=150, bbox_inches='tight')

## Loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(biased.loss_history, label='Biased')
ax.plot(joint.loss_history,  label='Joint')
ax.set_xlabel('Step'); ax.set_ylabel('SymKL loss')
ax.legend(); ax.set_title('Loss comparison')
fig.tight_layout()
fig.savefig('unknown_edges_loss.png', dpi=150, bbox_inches='tight')

## Gradient of the loss w.r.t. unknown entries

For reference, the analytical gradient of the SymKL loss w.r.t. an unknown edge weight $a_{mn}$ is:

$$\frac{\partial \mathcal{L}}{\partial a_{mn}} = H_{mn} + H_{nm} - H_{mm} - H_{nn}$$

where $H = Q G^\top Q$ and $G_{ij} = -\hat{A}_{ij}/Q_{ij} + \log(Q_{ij}/\hat{A}_{ij}) + 1$.

This is computed automatically by PyTorch autograd through the `forest_matrix` and `soft_decoder` calls.